# Explore the canonical Lamin collection safely

This notebook is a read-only tour of the current pert-gym canonical query surface on LaminDB:

- instance: `laminlabs/pertdata`
- branch: `jkobject`
- canonical collection: `pert-gym/canonical/20260621`
- base public collection: `pert-gym/base-public/20260621`
- pert-gym additions collection: `pert-gym/additions/20260621`
- model-ready collection: `pert-gym/model-ready/20260621`

Collection membership is artifact-level: canonical members are `obs.parquet` artifacts. The matrix and feature table are resolved through Lamin feature links: `obs -> X -> var`.

Safety rule: query manifest/metadata first, load `obs` and `var` deliberately, and load `X.h5ad` only for a deliberately tiny reviewed member. Do **not** call `.to_memory()` or full-load PRISM/VIPerturb/Cellarity matrices unless RAM/disk has been explicitly budgeted.


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display

from tools.lamin_context import connect_pertdata

DATE = "20260621"
BASE_PUBLIC_KEY = f"pert-gym/base-public/{DATE}"
ADDITIONS_KEY = f"pert-gym/additions/{DATE}"
CANONICAL_KEY = f"pert-gym/canonical/{DATE}"
MODEL_READY_KEY = f"pert-gym/model-ready/{DATE}"
MANIFEST_PATH = Path(f"artifacts/schema_audit/unified_collection_manifest_{DATE}.tsv")

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)


## 1. Connect to the pinned Lamin instance and branch

Always use the project helper. It pins `laminlabs/pertdata` and branch `jkobject` and redirects the Lamin cache under this repository.

This notebook is read-only and does not create Lamin artifacts, so it does not call `ln.track()`. If you adapt it to write artifacts, call `ln.track()` immediately after connecting and before writes.


In [ ]:
ln = connect_pertdata()
assert ln.setup.settings.instance.slug == "laminlabs/pertdata"
assert ln.setup.settings.branch.name == "jkobject"
print("connected", ln.setup.settings.instance.slug, ln.setup.settings.branch.name)


## 2. Locate the Collection family and count members

`canonical` is the default triplet-integrity query surface. `model-ready` is a smaller reviewed loader-smoked subset and must not be confused with the full canonical collection.


In [ ]:
collection_keys = {
    "base_public": BASE_PUBLIC_KEY,
    "additions": ADDITIONS_KEY,
    "canonical": CANONICAL_KEY,
    "model_ready": MODEL_READY_KEY,
}
collections = {role: ln.Collection.get(key=key) for role, key in collection_keys.items()}
collection_counts = {role: coll.artifacts.count() for role, coll in collections.items()}
collection_counts


In [ ]:
member_keys = {
    role: list(coll.artifacts.all().values_list("key", flat=True))
    for role, coll in collections.items()
}
for role, keys in member_keys.items():
    print(f"{role:>12}: {len(keys):5d} members")
print("\ncanonical examples:")
member_keys["canonical"][:10]


## 3. Load the local unified manifest for fast filtering

The manifest is the cheap exploration index. It records source family, logical dataset, chunking, obs/var sizes, branch provenance, harmonization level, and link-check flags.

The manifest is an artifact generated by the P3R collection build under `artifacts/schema_audit/`. If it is missing from a fresh checkout, run or fetch the P3R build artifacts before using this notebook; do not replace it by scanning all Lamin artifacts ad hoc.


In [ ]:
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Missing {MANIFEST_PATH}. Fetch or regenerate the P3R unified collection manifest first."
    )
manifest = pd.read_csv(MANIFEST_PATH, sep="	", keep_default_na=False)
for column in ["n_obs", "n_vars", "chunk_index", "harmonization_level_rank"]:
    if column in manifest.columns:
        manifest[column] = pd.to_numeric(manifest[column], errors="coerce")
for column in ["has_obs_x_link", "has_x_var_link", "same_prefix_var", "link_verification_checked"]:
    if column in manifest.columns:
        manifest[column] = manifest[column].map(lambda value: str(value).lower() == "true")
print(manifest.shape)
manifest.head(10)


## 4. Count vocabulary

These counters answer different questions. Do not collapse them into one "dataset count".


In [ ]:
count_vocabulary = {
    "canonical_collection_members_obs_artifacts": collection_counts["canonical"],
    "base_public_collection_members": collection_counts["base_public"],
    "pert_gym_addition_collection_members": collection_counts["additions"],
    "model_ready_members": collection_counts["model_ready"],
    "manifest_rows": len(manifest),
    "logical_datasets_or_families": manifest["logical_dataset"].nunique(),
    "triplet_prefixes_or_chunks": manifest["prefix"].nunique(),
    "chunked_members": int((manifest["split"] == "chunk").sum()),
    "non_chunk_members": int((manifest["split"] != "chunk").sum()),
    "harmonization_level_counts": manifest["harmonization_level"].value_counts().to_dict(),
}
count_vocabulary


In [ ]:
source_counts = manifest["source"].value_counts().rename_axis("source").reset_index(name="members")
modality_counts = manifest["modality"].value_counts().rename_axis("modality").reset_index(name="members")
display(source_counts.head(15))
display(modality_counts)


## 5. Metadata-only filtering examples

These queries operate only on the manifest. They are the first step before resolving or loading any payload. The examples use the vocabulary actually present in the manifest, including currently coarse `organism` and `perturbation_type` values.


In [ ]:
organism_counts = manifest["organism"].value_counts(dropna=False)
perturbation_type_counts = manifest["perturbation_type"].value_counts(dropna=False)
print("organism vocabulary:", organism_counts.to_dict())
print("perturbation_type vocabulary:", perturbation_type_counts.to_dict())

# Current manifest is conservative: organism is still `unknown` for all rows, and
# perturbation types are coarse (`CRISPR`, `drug`, `unknown`). Use the vocabulary
# as recorded rather than inventing finer labels.
unknown_organism_scrna = manifest.query("organism == 'unknown' and modality == 'scRNA-seq'").copy()
crispr_members = manifest.query("perturbation_type == 'CRISPR'").copy()
drug_members = manifest.query("perturbation_type == 'drug'").copy()
prism_chunks = manifest.query("source == 'PRISM' and split == 'chunk'").copy()
small_obs_members = manifest[manifest["n_obs"].fillna(10**18) <= 5_000].copy()

print("unknown-organism scRNA-seq", unknown_organism_scrna.shape)
print("CRISPR perturbation members", crispr_members.shape)
print("drug perturbation members", drug_members.shape)
print("PRISM chunks", prism_chunks.shape)
print("small obs members <= 5k", small_obs_members.shape)

unknown_organism_scrna[["source", "dataset_id", "artifact_key", "n_obs", "n_vars", "perturbation_type", "control_availability"]].head(10)


In [ ]:
prism_chunks[["dataset_id", "artifact_key", "chunk_id", "n_obs", "n_vars", "control_availability", "harmonization_level"]].head(10)


## 6. Resolve `obs -> X -> var` links

Lamin feature values may be returned either as Artifact objects or as artifact key strings, so the resolver handles both.


In [ ]:
def resolve_artifact(value: Any):
    if value is None:
        raise KeyError("missing Lamin feature link")
    if isinstance(value, str):
        return ln.Artifact.get(key=value)
    if getattr(value, "key", None):
        return value
    raise TypeError(f"Unsupported feature-link value: {type(value)!r}")


def resolve_triplet(artifact_key: str) -> dict[str, Any]:
    obs = ln.Artifact.get(key=artifact_key)
    x = resolve_artifact(obs.features.get_values()["X"])
    var = resolve_artifact(x.features.get_values()["var"])
    return {
        "obs_artifact": obs,
        "x_artifact": x,
        "var_artifact": var,
        "obs_key": obs.key,
        "x_key": x.key,
        "var_key": var.key,
        "obs_n_observations": getattr(obs, "n_observations", None),
        "x_n_observations": getattr(x, "n_observations", None),
    }

public_example_key = manifest.query("collection_category == 'base_public'").sort_values(
    ["n_obs", "n_vars"], na_position="last"
).iloc[0]["artifact_key"]
prism_example_key = prism_chunks.sort_values(["n_obs", "chunk_index"], na_position="last").iloc[0]["artifact_key"]
model_ready_key = member_keys["model_ready"][0]

examples = {
    "public_base_smallest": public_example_key,
    "prism_chunk_metadata_only": prism_example_key,
    "model_ready_tiny_reviewed": model_ready_key,
}
examples


In [ ]:
resolved = {label: resolve_triplet(key) for label, key in examples.items()}
pd.DataFrame(
    [
        {
            "label": label,
            "obs_key": triplet["obs_key"],
            "x_key": triplet["x_key"],
            "var_key": triplet["var_key"],
            "obs_n_observations": triplet["obs_n_observations"],
            "x_n_observations": triplet["x_n_observations"],
        }
        for label, triplet in resolved.items()
    ]
)


## 7. Load small `obs` and `var` metadata only

This cell loads only `obs.parquet` and `var.parquet` for selected small/reviewed examples. For very large obs tables, prefer column/row-limited parquet inspection or a manifest-only pass.


In [ ]:
CANONICAL_OBS_COLUMNS = [
    "perturbation",
    "perturbation_type",
    "perturbation_technology",
    "organism",
    "cell_line",
    "cell_type",
    "modality",
    "assay",
    "is_control",
    "timepoint",
    "dose",
]

metadata_summaries = []
for label in ["public_base_smallest", "model_ready_tiny_reviewed"]:
    triplet = resolved[label]
    obs_rows = int(triplet["obs_n_observations"] or 10**18)
    if obs_rows > 5_000:
        raise RuntimeError(f"Refusing to load obs for {label}: n_obs={obs_rows}")
    obs_df = triplet["obs_artifact"].load()
    var_df = triplet["var_artifact"].load()
    metadata_summaries.append(
        {
            "label": label,
            "obs_shape": obs_df.shape,
            "var_shape": var_df.shape,
            "canonical_obs_columns_present": [col for col in CANONICAL_OBS_COLUMNS if col in obs_df.columns],
            "obs_columns_sample": obs_df.columns[:20].tolist(),
            "var_columns_sample": var_df.columns[:12].tolist(),
        }
    )

pd.DataFrame(metadata_summaries)


## 8. Safe `X.h5ad` proof on the tiny reviewed model-ready member only

This is the only cell that loads an `X.h5ad` payload. It is guarded by `n_obs <= 5_000` and uses the 1-member `model-ready` reviewed subset, not a large PRISM/VIPerturb chunk.


In [ ]:
small_triplet = resolved["model_ready_tiny_reviewed"]
small_n_obs = int(small_triplet["obs_n_observations"] or small_triplet["x_n_observations"] or 10**18)
if small_n_obs > 5_000:
    raise RuntimeError(f"Refusing safe X load for {small_triplet['x_key']} with n_obs={small_n_obs}")

small_adata = small_triplet["x_artifact"].load()
print("x_key:", small_triplet["x_key"])
print("shape:", small_adata.shape)
print("loaded reviewed model-ready member only; keep PRISM/VIPerturb large chunks metadata/backed-only")


## 9. Large-member pattern: metadata/backed-only, no matrix materialization

For large PRISM/VIPerturb members, stop at manifest rows plus artifact handles unless a separate ingestion/modeling task explicitly budgets memory. If you need to inspect a remote `X.h5ad`, prefer backed reads against a verified local cache path and record the RAM/disk budget first.


In [ ]:
large_candidates = manifest.sort_values("n_obs", ascending=False).head(10)[
    ["source", "dataset_id", "artifact_key", "prefix", "n_obs", "n_vars", "split", "chunk_id"]
]
large_candidates


In [ ]:
large_example_key = large_candidates.iloc[0]["artifact_key"]
large_triplet = resolve_triplet(large_example_key)
{
    "obs_key": large_triplet["obs_key"],
    "x_key": large_triplet["x_key"],
    "var_key": large_triplet["var_key"],
    "x_n_observations": large_triplet["x_n_observations"],
    "note": "Artifact handles resolved; X not loaded in this notebook.",
}


## 10. Takeaways

- Use `tools.lamin_context.connect_pertdata()` and assert `laminlabs/pertdata` / `jkobject`.
- Use `pert-gym/canonical/20260621` as the unified obs-member surface.
- Use `artifacts/schema_audit/unified_collection_manifest_20260621.tsv` for cheap filtering by `source`, `modality`, `organism`, `perturbation_type`, `logical_dataset`, `split`, `harmonization_level`, and size.
- Resolve payloads through `obs.features['X']` and `X.features['var']`; do not assume all triplet parts are collection members.
- Keep count vocabulary explicit: artifact records, collection members, logical datasets/families, triplet prefixes/chunks, and model-ready members are different counters.
- Treat `model-ready` as a reviewed loader-smoked subset (currently 1 member), not as the full canonical query surface.
